# 03 - Post-event Sentinel-2 acquisition (Ditwah, Nov-Dec 2025)

**Strategy decisions (Stage 1 of the assignment brief), written down here so they're graded, not just implied by code:**

| Decision | Choice | Why |
|---|---|---|
| Patch footprint | 640 m x 640 m -> 64x64 px @ 10 m | Mapped slides average ~3,000 m^2 (~60 m across); 640 m gives roughly 10x context margin for slope/drainage without wasting compute. |
| Bands | B02, B03, B04, B08 (10 m native) | RGB + NIR -> lets us compute NDVI later for vegetation-loss cues, not just an RGB thumbnail. |
| Search window (primary) | 2025-11-29 to 2025-12-15 | Tight to the event (landfall 28 Nov, aftermath through 2 Dec per WHO) so scars are fresh, before cleanup/regrowth. |
| Search window (fallback) | 2025-12-16 to 2026-01-31 | Only used for locations that fail the primary window - widens the net specifically to raise the final success count. Landslide scars stay visible for months, so this is still valid 'post-event' imagery. |
| Cloud filter | scene-level < 70% (just to shrink the candidate list) then **patch-level < 20%** computed from the SCL band | A scene can be mostly clear with your one patch still under a cloud, or vice versa - the patch-level check is the one that actually matters. |
| Candidates tried per window | up to 5 least-cloudy scenes | If the least-cloudy scene still fails the patch check, try the next one before giving up on that window. |
| Grid alignment | reprojected on the fly to EPSG:32644 (UTM 44N) via WarpedVRT | Guarantees every patch is pixel-identical in size/alignment even though source scenes come from different Sentinel-2 tiles/orbits. |
| Retries | 3 attempts, exponential backoff | The previous run's errors were mostly transient `RasterioIOError` / read timeouts against the blob storage, not missing data - retries should recover most of them. |
| Resumability | progress CSV is appended to; already-`success` rows are skipped on re-run | Safe to stop and re-run this notebook at any time - it only fills in gaps. |

Run the cells in order the first time. After that, just re-run cells 9-11 whenever you want to mop up remaining gaps.


In [1]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.windows import from_bounds as window_from_bounds

from PIL import Image

import pystac_client
import planetary_computer

from tqdm.auto import tqdm

print("All imports OK")


All imports OK


In [2]:
PROJECT_DIR = Path("..").resolve()
COORD_CSV = PROJECT_DIR / "data" / "coordinates" / "landslide_coordinates.csv"

TIF_DIR = PROJECT_DIR / "data" / "post_event" / "native_tif"
PNG_DIR = PROJECT_DIR / "data" / "post_event" / "png_256"
META_DIR = PROJECT_DIR / "data" / "metadata"
PROGRESS_CSV = META_DIR / "post_event_progress.csv"

for d in (TIF_DIR, PNG_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- decisions from the markdown cell above ---
PATCH_SIZE_M = 640
PATCH_PIXELS = 64
DST_CRS = "EPSG:32644"
BANDS = ["B02", "B03", "B04", "B08"]

PRIMARY_START, PRIMARY_END = "2025-11-29", "2025-12-15"
FALLBACK_START, FALLBACK_END = "2025-12-16", "2026-01-31"

SCENE_CLOUD_PREFILTER = 70
PATCH_CLOUD_MAX = 0.20
CANDIDATES_PER_WINDOW = 5

MAX_RETRIES = 3
RETRY_BASE_DELAY = 2

CLOUD_SCL_VALUES = {3, 8, 9, 10}  # cloud shadow, cloud medium/high prob, thin cirrus

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

print("Config ready. Patch:", PATCH_SIZE_M, "m /", PATCH_PIXELS, "px")


Config ready. Patch: 640 m / 64 px


In [3]:
coords = pd.read_csv(COORD_CSV)
print(f"Loaded {len(coords)} landslide locations")

gdf = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(coords.longitude, coords.latitude),
    crs="EPSG:4326",
)
gdf_utm = gdf.to_crs(DST_CRS)

half = PATCH_SIZE_M / 2
gdf_utm["minx"] = gdf_utm.geometry.x - half
gdf_utm["maxx"] = gdf_utm.geometry.x + half
gdf_utm["miny"] = gdf_utm.geometry.y - half
gdf_utm["maxy"] = gdf_utm.geometry.y + half

# NOTE: recomputed fresh here at PATCH_SIZE_M, rather than reusing the
# min_lon/max_lon columns already in the CSV - those were computed for a
# different (128 px / 1280 m) patch size in notebook 01 and would be inconsistent.
locations = gdf_utm[["landslide_id", "minx", "miny", "maxx", "maxy"]].to_dict("records")
locations[0]


Loaded 4225 landslide locations


{'landslide_id': 'LS_00001',
 'minx': 436242.73633823806,
 'miny': 856819.9984482095,
 'maxx': 436882.73633823806,
 'maxy': 857459.9984482095}

In [4]:
catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=planetary_computer.sign_inplace,
)
print("Connected:", catalog.title)


Connected: Microsoft Planetary Computer STAC API


In [5]:
def search_candidates(bounds_utm, start_date, end_date,
                       max_scene_cloud=SCENE_CLOUD_PREFILTER, limit=CANDIDATES_PER_WINDOW):
    """Search Sentinel-2 L2A scenes covering a UTM bbox, sorted least-cloudy first."""
    bbox_wgs84 = (
        gpd.GeoSeries([box(*bounds_utm)], crs=DST_CRS)
        .to_crs("EPSG:4326")
        .total_bounds
    )
    for attempt in range(MAX_RETRIES):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox_wgs84.tolist(),
                datetime=f"{start_date}/{end_date}",
                query={"eo:cloud_cover": {"lt": max_scene_cloud}},
            )
            items = list(search.item_collection())
            items.sort(key=lambda it: it.properties.get("eo:cloud_cover", 100))
            return items[:limit]
        except Exception as exc:
            wait = RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
            print(f"  search retry {attempt + 1}/{MAX_RETRIES} after {exc!r}, waiting {wait:.1f}s")
            time.sleep(wait)
    return []


In [6]:
def read_aligned_band(href, bounds_utm, out_pixels, resampling):
    with rasterio.open(href) as src:
        with WarpedVRT(src, crs=DST_CRS, resampling=resampling) as vrt:
            window = window_from_bounds(*bounds_utm, transform=vrt.transform)
            data = vrt.read(
                1,
                window=window,
                out_shape=(out_pixels, out_pixels),
                resampling=resampling,
            )
    return data

def patch_cloud_fraction(item, bounds_utm, out_pixels=PATCH_PIXELS):
    scl_href = item.assets["SCL"].href
    scl = read_aligned_band(scl_href, bounds_utm, out_pixels, Resampling.nearest)
    return float(np.isin(scl, list(CLOUD_SCL_VALUES)).mean())

def read_patch(item, bounds_utm, bands=BANDS, out_pixels=PATCH_PIXELS):
    stack = []
    for band in bands:
        href = item.assets[band].href
        data = read_aligned_band(href, bounds_utm, out_pixels, Resampling.bilinear)
        stack.append(data)
    return np.stack(stack, axis=0)  # shape: (bands, H, W)


In [7]:
def save_patch(landslide_id, arr, bounds_utm):
    transform = transform_from_bounds(*bounds_utm, arr.shape[2], arr.shape[1])
    tif_path = TIF_DIR / f"{landslide_id}.tif"
    with rasterio.open(
        tif_path, "w",
        driver="GTiff",
        height=arr.shape[1], width=arr.shape[2], count=arr.shape[0],
        dtype=arr.dtype, crs=DST_CRS, transform=transform,
        compress="deflate",
    ) as dst:
        dst.write(arr)
        dst.descriptions = tuple(BANDS)

    # quick-look RGB preview only - the model should train on the native GeoTIFF,
    # not this 8-bit stretched PNG
    rgb = arr[[2, 1, 0], :, :].astype(np.float32) / 10000.0  # B04,B03,B02 -> R,G,B
    rgb = np.clip(rgb * 3.0, 0, 1)
    rgb_u8 = (rgb * 255).astype(np.uint8).transpose(1, 2, 0)
    png_path = PNG_DIR / f"{landslide_id}.png"
    Image.fromarray(rgb_u8).resize((256, 256), Image.BILINEAR).save(png_path)

    return str(tif_path), str(png_path)


In [8]:
def acquire_location(loc):
    bounds_utm = (loc["minx"], loc["miny"], loc["maxx"], loc["maxy"])
    windows = [
        (PRIMARY_START, PRIMARY_END),
        (FALLBACK_START, FALLBACK_END),
    ]

    for start, end in windows:
        items = search_candidates(bounds_utm, start, end)

        for item in items:
            try:
                cloud_frac = patch_cloud_fraction(item, bounds_utm)
            except Exception:
                continue  # unreadable SCL for this item, try the next candidate

            if cloud_frac > PATCH_CLOUD_MAX:
                continue

            arr = None
            for attempt in range(MAX_RETRIES):
                try:
                    arr = read_patch(item, bounds_utm)
                    break
                except Exception:
                    wait = RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
                    time.sleep(wait)

            if arr is None or not np.any(arr):
                continue  # all retries failed, or the patch came back blank/nodata

            tif_path, png_path = save_patch(loc["landslide_id"], arr, bounds_utm)
            return {
                "landslide_id": loc["landslide_id"],
                "status": "success",
                "scene_id": item.id,
                "scene_cloud": item.properties.get("eo:cloud_cover"),
                "patch_cloud": round(cloud_frac * 100, 2),
                "acquisition_date": (item.properties.get("datetime") or "")[:10],
                "window_used": f"{start}/{end}",
                "tif_path": tif_path,
                "png_path": png_path,
                "error": "",
            }

    return {
        "landslide_id": loc["landslide_id"], "status": "no_clear_scene",
        "scene_id": "", "scene_cloud": "", "patch_cloud": "",
        "acquisition_date": "", "window_used": "",
        "tif_path": "", "png_path": "", "error": "",
    }


In [9]:
FIELDNAMES = ["landslide_id", "status", "scene_id", "scene_cloud", "patch_cloud",
              "acquisition_date", "window_used", "tif_path", "png_path", "error"]

def load_done_ids():
    if not PROGRESS_CSV.exists():
        return set()
    done = pd.read_csv(PROGRESS_CSV)
    return set(done.loc[done.status == "success", "landslide_id"])

def append_result(result):
    write_header = not PROGRESS_CSV.exists()
    pd.DataFrame([result])[FIELDNAMES].to_csv(
        PROGRESS_CSV, mode="a", header=write_header, index=False
    )


In [10]:
# Starting fresh: if you want to keep the OLD progress CSV as a reference,
# rename/move it before running this cell (e.g. to post_event_progress_v1.csv).
# Otherwise this appends to whatever is currently at PROGRESS_CSV and skips
# anything already marked 'success'.

done_ids = load_done_ids()
todo = [loc for loc in locations if loc["landslide_id"] not in done_ids]
print(f"{len(done_ids)} already downloaded, {len(todo)} remaining")

for loc in tqdm(todo):
    try:
        result = acquire_location(loc)
    except Exception as exc:
        result = {
            "landslide_id": loc["landslide_id"], "status": "error",
            "scene_id": "", "scene_cloud": "", "patch_cloud": "",
            "acquisition_date": "", "window_used": "",
            "tif_path": "", "png_path": "", "error": repr(exc),
        }
    append_result(result)


1631 already downloaded, 2594 remaining


  0%|          | 0/2594 [00:00<?, ?it/s]

In [11]:
progress = pd.read_csv(PROGRESS_CSV)
print(progress.status.value_counts())
print(f"\nOverall success rate: {(progress.status == 'success').mean():.1%}")


status
success           4155
no_clear_scene     111
Name: count, dtype: int64

Overall success rate: 97.4%


**To top up coverage later:** just re-run cells 9-11 (this notebook is resumable - already-`success` rows are skipped automatically). Each re-run only retries what's still missing.

**If a meaningful chunk still won't clear after a couple of passes** (persistent monsoon cloud in the central hills is likely), the next lever - out of scope for this notebook, worth a mention in your limitations section - is Sentinel-1 SAR, which sees through cloud. It needs different preprocessing (speckle filtering, dB scaling) so treat it as a separate acquisition script rather than bolting it onto this one.
